# Node2Vec benchmark: mlx-graphs vs PyTorch Geometric

This notebook compares Node2Vec in **mlx-graphs** (walks from `mlx_cluster`, running on MLX) with **PyTorch Geometric** (walks from `pyg-lib`, running on PyTorch) on the Cora graph.

It measures three things:

1. **Walk sampling**: the random-walk kernels alone, for uniform walks (`p = q = 1`) and biased node2vec walks (`p`/`q` ≠ 1).
2. **Training**: seconds per epoch for each library's full `Node2Vec` training loop.
3. **Embedding quality**: logistic-regression test accuracy on Cora, so speed is compared at equal quality.

> [!NOTE]
> PyG 2.8's `Node2Vec` samples walks with `pyg-lib`, whose `random_walk` currently supports **uniform walks only** (`p = q = 1`); biased walks raise `Uniform sampling required for now`. Biased-walk rows therefore show PyG as "not supported" and report mlx-graphs alone.
>
> On macOS, `pyg-lib` ships CPU builds only, so PyG's walks always run on the CPU (PyG's `Node2Vec` keeps the graph on the CPU). PyG's embedding training can still use the GPU through MPS. On the MLX side, walks run on the GPU with Metal.

**Requirements:** `mlx_cluster>=0.1.2` (or a source build of current `main`), mlx-graphs with the Node2Vec `rand_data` fix, `torch`, `torch_geometric>=2.8`, `pyg-lib`, `scikit-learn`.

## Setup

In [1]:
import platform
import statistics
import time
from importlib.metadata import PackageNotFoundError, version

import mlx.core as mx
import mlx.nn as mxnn
import mlx.optimizers as mxoptim
import numpy as np
import torch
import torch_geometric
from sklearn.linear_model import LogisticRegression
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import Node2Vec as PyGNode2Vec
from torch_geometric.typing import WITH_PYG_LIB

import mlx_cluster
from mlx_graphs.algorithms import Node2Vec as MXGNode2Vec

def pkg_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "not installed"

print(f"Python          {platform.python_version()} ({platform.machine()})")
for name in ["mlx", "mlx_cluster", "mlx-graphs", "torch", "torch_geometric", "pyg-lib", "scikit-learn"]:
    print(f"{name:<15} {pkg_version(name)}")
print()
print("MLX Metal GPU available:", mx.metal.is_available())
print("PyTorch MPS available:  ", torch.backends.mps.is_available())
print("PyG has pyg-lib:        ", WITH_PYG_LIB)
assert WITH_PYG_LIB, "PyG's Node2Vec needs pyg-lib (see the install instructions)"

/Users/vinay.pandya/mlx_graphs_workspace/mlx_graphs_venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/vinay.pandya/mlx_graphs_workspace/mlx_graphs_venv/lib/python3.13/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.26.4 and <2.7.0 is required for this version of SciPy (detected version 1.26.3)
  from scipy.sparse import csr_matrix, issparse


Python          3.13.5 (arm64)
mlx             0.32.4
mlx_cluster     0.1.2
mlx-graphs      0.0.10
torch           2.12.1
torch_geometric 2.8.1
pyg-lib         0.9.0+pt212
scikit-learn    1.8.0

MLX Metal GPU available: True
PyTorch MPS available:   True
PyG has pyg-lib:         True


In [2]:
import inspect

# rejection_sampling has a GPU kernel only in mlx_cluster >= 0.1.2 (it added max_attempts= at the same time)
MLX_CLUSTER_RS_HAS_GPU = "max_attempts" in (mlx_cluster.rejection_sampling.__doc__ or "")
# Upstream mlx-graphs hard-codes stream=mx.cpu for biased walks; a local edit may enable the GPU
MXG_BIASED_WALKS_ON_GPU = "stream=mx.cpu," not in inspect.getsource(MXGNode2Vec.pos_sample)

def _pyg_supports_biased_walks():
    """pyg-lib's random_walk raises for p/q != 1 ("Uniform sampling required for now")."""
    rowptr, col = torch.tensor([0, 1, 2]), torch.tensor([1, 0])
    try:
        torch.ops.pyg.random_walk(rowptr, col, torch.tensor([0]), 2, 1.0, 0.5)
        return True
    except RuntimeError:
        return False

PYG_SUPPORTS_BIASED = _pyg_supports_biased_walks()

print("mlx_cluster rejection_sampling has GPU kernel:", MLX_CLUSTER_RS_HAS_GPU)
print("mlx-graphs Node2Vec runs biased walks on GPU:  ", MXG_BIASED_WALKS_ON_GPU)
print("PyG (pyg-lib) supports biased walks:           ", PYG_SUPPORTS_BIASED)

def is_biased(p, q):
    return not (p == 1.0 and q == 1.0)

def pyg_supports(p, q):
    return PYG_SUPPORTS_BIASED or not is_biased(p, q)

mlx_cluster rejection_sampling has GPU kernel: True
mlx-graphs Node2Vec runs biased walks on GPU:   True
PyG (pyg-lib) supports biased walks:            False


## Configuration

Both libraries use the same graph and the same hyperparameters.

In [3]:
# Training runs are repeated for each seed; results are reported as mean ± std
SEEDS = [0, 1, 2]

# Node2Vec hyperparameters (shared by both libraries)
EMBEDDING_DIM = 128
WALK_LENGTH = 20
CONTEXT_SIZE = 10
WALKS_PER_NODE = 10
NUM_NEGATIVE_SAMPLES = 1
BATCH_SIZE = 128
LEARNING_RATE = 0.01
NUM_EPOCHS = 10   # raise (e.g. to 50) to compare converged accuracy

# MLX's Adam defaults to bias_correction=False, PyTorch's Adam always applies it.
# With MATCH_ADAM, MLX uses bias correction too, so both optimizers behave the same.
MATCH_ADAM = True

# PyTorch initializes nn.Embedding with std 1.0, MLX with std 1/sqrt(dim). With MATCH_INIT, PyG's
# embeddings start at MLX's scale so both models begin from the same point; set False for library defaults.
MATCH_INIT = True

# Walk settings to benchmark: (label, p, q)
WALK_SETTINGS = [("uniform (p=1, q=1)", 1.0, 1.0), ("biased (p=1, q=0.5)", 1.0, 0.5)]

# Walk-sampling benchmark
SAMPLING_WALK_LENGTH = 80   # typical node2vec walk length
WARMUP, REPEATS = 2, 10

TORCH_DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("PyTorch training device:", TORCH_DEVICE)

PyTorch training device: mps


## Dataset

Cora is loaded once through PyG, and the **same** `edge_index` is passed to both libraries, so both work on an identical graph.

In [4]:
dataset = Planetoid(root="data/Planetoid", name="Cora")
data = dataset[0]
num_nodes = data.num_nodes

edge_index_torch = data.edge_index                       # int64, shape (2, E)
edge_index_mx = mx.array(data.edge_index.numpy())        # same edges as an MLX array

print(f"Cora: {num_nodes} nodes, {data.num_edges} edges")

Cora: 2708 nodes, 10556 edges


## Helpers

In [5]:
def benchmark(fn, sync, warmup=WARMUP, repeats=REPEATS):
    """Median wall-clock time of fn() in milliseconds. sync(result) forces lazy work to finish."""
    for _ in range(warmup):            # first calls compile Metal kernels / warm caches
        sync(fn())
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        sync(fn())
        times.append(time.perf_counter() - start)
    return statistics.median(times) * 1000

def mx_sync(out):
    mx.eval(out)

def torch_sync(out):
    if TORCH_DEVICE.type == "mps":
        torch.mps.synchronize()

def print_table(header, rows):
    widths = [max(len(str(r[i])) for r in [header] + rows) for i in range(len(header))]
    line = lambda r: "| " + " | ".join(str(c).ljust(w) for c, w in zip(r, widths)) + " |"
    print(line(header))
    print("|" + "|".join("-" * (w + 2) for w in widths) + "|")
    for r in rows:
        print(line(r))

## 1. Walk sampling

This benchmark times the random-walk kernels directly, with the same CSR graph and the same start nodes (every node × `WALKS_PER_NODE`), so it doesn't depend on either library's training code.

- **mlx_cluster:** `random_walk` for uniform walks, `rejection_sampling` for biased walks, on GPU and CPU.
- **PyG:** `torch.ops.pyg.random_walk` (pyg-lib), CPU only on macOS.

In [6]:
# Build CSR once from PyG's sorted graph and share it with MLX
pyg_ref = PyGNode2Vec(edge_index_torch, embedding_dim=8, walk_length=2, context_size=1, num_nodes=num_nodes)
rowptr_t, col_t = pyg_ref.rowptr, pyg_ref.col
rowptr_m, col_m = mx.array(rowptr_t.numpy()), mx.array(col_t.numpy())

start_t = torch.arange(num_nodes).repeat(WALKS_PER_NODE)
start_m = mx.array(start_t.numpy())
num_walks = start_t.numel()
L = SAMPLING_WALK_LENGTH
mx.eval(rowptr_m, col_m, start_m)
print(f"{num_walks:,} walks of length {L}")

def pyg_walks(p, q):
    out = torch.ops.pyg.random_walk(rowptr_t, col_t, start_t, L, p, q)
    return out if isinstance(out, torch.Tensor) else out[0]

def mlx_walks(p, q, stream):
    if p == 1.0 and q == 1.0:
        rand = mx.random.uniform(shape=[num_walks, L])
        return mlx_cluster.random_walk(rowptr_m, col_m, start_m, rand, L, stream=stream)[0]
    return mlx_cluster.rejection_sampling(rowptr_m, col_m, start_m, L, p, q, stream=stream)[0]

def mlx_gpu_available(p, q):
    return mx.metal.is_available() and (not is_biased(p, q) or MLX_CLUSTER_RS_HAS_GPU)

sampling_rows = []
for label, p, q in WALK_SETTINGS:
    t_pyg = benchmark(lambda: pyg_walks(p, q), sync=lambda out: None) if pyg_supports(p, q) else None
    t_mlx_cpu = benchmark(lambda: mlx_walks(p, q, mx.cpu), sync=mx_sync)
    t_mlx_gpu = benchmark(lambda: mlx_walks(p, q, mx.gpu), sync=mx_sync) if mlx_gpu_available(p, q) else None
    sampling_rows.append([
        label,
        f"{t_pyg:.1f}" if t_pyg is not None else "not supported",
        f"{t_mlx_cpu:.1f}",
        f"{t_mlx_gpu:.1f}" if t_mlx_gpu is not None else "n/a",
        f"{t_pyg / t_mlx_gpu:.1f}x" if t_pyg is not None and t_mlx_gpu is not None else "n/a",
    ])

print_table(["Walks", "PyG CPU (ms)", "MLX CPU (ms)", "MLX GPU (ms)", "MLX GPU vs PyG"], sampling_rows)

27,080 walks of length 80
| Walks               | PyG CPU (ms)  | MLX CPU (ms) | MLX GPU (ms) | MLX GPU vs PyG |
|---------------------|---------------|--------------|--------------|----------------|
| uniform (p=1, q=1)  | 18.0          | 15.8         | 2.4          | 7.4x           |
| biased (p=1, q=0.5) | not supported | 52.4         | 6.1          | n/a            |


### Sanity check: walks follow graph edges

Speed only counts if the walks are valid, so this cell checks that every step of each sampler's walks is an edge of the graph (or a stay-in-place on a node with no outgoing edges).

In [7]:
rowptr_np, col_np = rowptr_t.numpy(), col_t.numpy()
edges = {(u, v) for u in range(num_nodes) for v in col_np[rowptr_np[u]:rowptr_np[u + 1]]}
has_out = {u for u, _ in edges}

def invalid_steps(walks, max_walks=2000):
    bad = 0
    for w in np.asarray(walks)[:max_walks]:
        for u, v in zip(w[:-1], w[1:]):
            if (u in has_out and (u, v) not in edges) or (u not in has_out and u != v):
                bad += 1
    return bad

for label, p, q in WALK_SETTINGS:
    checks = {"MLX CPU": lambda: mlx_walks(p, q, mx.cpu)}
    if pyg_supports(p, q):
        checks["PyG"] = lambda: pyg_walks(p, q).numpy()
    if mlx_gpu_available(p, q):
        checks["MLX GPU"] = lambda: mlx_walks(p, q, mx.gpu)
    for name, sample in checks.items():
        print(f"{label:<20} {name:<8} {invalid_steps(sample())} invalid steps")

uniform (p=1, q=1)   MLX CPU  0 invalid steps
uniform (p=1, q=1)   PyG      0 invalid steps
uniform (p=1, q=1)   MLX GPU  0 invalid steps
biased (p=1, q=0.5)  MLX CPU  0 invalid steps
biased (p=1, q=0.5)  MLX GPU  0 invalid steps


## 2. Training

Each library trains its own `Node2Vec` model for `NUM_EPOCHS` epochs with dense Adam (same learning rate and batch size), once per seed in `SEEDS`. The timer covers walk sampling, the loss, gradients and optimizer updates. After training, each run's embeddings are scored with logistic regression on Cora (section 3).

- **mlx-graphs:** walks on the GPU (biased walks only if `MXG_BIASED_WALKS_ON_GPU` is `True`), training on MLX's default device (GPU).
- **PyG:** walks on the CPU (pyg-lib), training on `TORCH_DEVICE` (MPS if available).

Two switches make the runs comparable (both on by default):
- `MATCH_INIT`: PyG's embeddings are initialized at MLX's scale (std 1/√`EMBEDDING_DIM` instead of PyTorch's 1.0).
- `MATCH_ADAM`: MLX's Adam uses bias correction, as PyTorch's Adam always does.

PyG is skipped for biased walks, which pyg-lib doesn't support.

In [8]:
train_mask, test_mask = data.train_mask.numpy(), data.test_mask.numpy()
labels = data.y.numpy()

def test_accuracy(embeddings):
    clf = LogisticRegression(max_iter=1000)
    clf.fit(embeddings[train_mask], labels[train_mask])
    return clf.score(embeddings[test_mask], labels[test_mask])

def train_mlx(p, q, seed):
    mx.random.seed(seed)
    np.random.seed(seed)  # mlx-graphs' dataloader shuffles with NumPy
    model = MXGNode2Vec(
        edge_index=edge_index_mx, embedding_dim=EMBEDDING_DIM, walk_length=WALK_LENGTH,
        context_size=CONTEXT_SIZE, num_nodes=num_nodes, walks_per_node=WALKS_PER_NODE,
        p=p, q=q, num_negative_samples=NUM_NEGATIVE_SAMPLES, use_gpu=mx.metal.is_available(),
    )
    optimizer = mxoptim.Adam(learning_rate=LEARNING_RATE, bias_correction=MATCH_ADAM)
    loss_and_grad = mxnn.value_and_grad(model, model.loss)
    epoch_times, losses = [], []
    for _ in range(NUM_EPOCHS):
        start, total, n = time.perf_counter(), 0.0, 0
        for pos, neg in model.dataloader(batch_size=BATCH_SIZE):
            loss, grads = loss_and_grad(pos, neg)
            optimizer.update(model, grads)
            mx.eval(model.parameters(), optimizer.state, loss)
            total, n = total + loss.item(), n + 1
        epoch_times.append(time.perf_counter() - start)
        losses.append(total / n)
    accuracy = test_accuracy(np.asarray(model.embedding.weight))
    return epoch_times, losses, accuracy

def train_pyg(p, q, seed):
    torch.manual_seed(seed)
    model = PyGNode2Vec(
        edge_index_torch, embedding_dim=EMBEDDING_DIM, walk_length=WALK_LENGTH,
        context_size=CONTEXT_SIZE, walks_per_node=WALKS_PER_NODE, p=p, q=q,
        num_negative_samples=NUM_NEGATIVE_SAMPLES, num_nodes=num_nodes, sparse=False,
    )
    if MATCH_INIT:
        torch.nn.init.normal_(model.embedding.weight, std=EMBEDDING_DIM ** -0.5)
    model = model.to(TORCH_DEVICE)
    loader = model.loader(batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    epoch_times, losses = [], []
    model.train()
    for _ in range(NUM_EPOCHS):
        start, total, n = time.perf_counter(), 0.0, 0
        for pos_rw, neg_rw in loader:
            optimizer.zero_grad()
            loss = model.loss(pos_rw.to(TORCH_DEVICE), neg_rw.to(TORCH_DEVICE))
            loss.backward()
            optimizer.step()
            total, n = total + loss.item(), n + 1
        torch_sync(None)
        epoch_times.append(time.perf_counter() - start)
        losses.append(total / n)
    accuracy = test_accuracy(model.embedding.weight.detach().cpu().numpy())
    return epoch_times, losses, accuracy

# results[(label, library)] -> list of (epoch_times, losses, accuracy), one per seed
results = {}
for label, p, q in WALK_SETTINGS:
    runs = [("mlx-graphs", train_mlx)] + ([("PyG", train_pyg)] if pyg_supports(p, q) else [])
    for library, train in runs:
        results[(label, library)] = []
        for seed in SEEDS:
            print(f"Training {library:<10} {label:<20} seed {seed}")
            results[(label, library)].append(train(p, q, seed))

Training mlx-graphs uniform (p=1, q=1)   seed 0


/Users/vinay.pandya/mlx_graphs_workspace/mlx_graphs_venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:206: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/vinay.pandya/mlx_graphs_workspace/mlx_graphs_venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:206: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/vinay.pandya/mlx_graphs_workspace/mlx_graphs_venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:206: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/vinay.pandya/mlx_graphs_workspace/mlx_graphs_venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:339: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_r

Training mlx-graphs uniform (p=1, q=1)   seed 1


/Users/vinay.pandya/mlx_graphs_workspace/mlx_graphs_venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:206: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/vinay.pandya/mlx_graphs_workspace/mlx_graphs_venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:206: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/vinay.pandya/mlx_graphs_workspace/mlx_graphs_venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:206: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/vinay.pandya/mlx_graphs_workspace/mlx_graphs_venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:339: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_r

Training mlx-graphs uniform (p=1, q=1)   seed 2


/Users/vinay.pandya/mlx_graphs_workspace/mlx_graphs_venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:206: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/vinay.pandya/mlx_graphs_workspace/mlx_graphs_venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:206: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/vinay.pandya/mlx_graphs_workspace/mlx_graphs_venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:206: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/vinay.pandya/mlx_graphs_workspace/mlx_graphs_venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:339: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_r

... (8 outputs left out)

Training mlx-graphs biased (p=1, q=0.5)  seed 1


/Users/vinay.pandya/mlx_graphs_workspace/mlx_graphs_venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:206: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/vinay.pandya/mlx_graphs_workspace/mlx_graphs_venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:206: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/vinay.pandya/mlx_graphs_workspace/mlx_graphs_venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:206: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/vinay.pandya/mlx_graphs_workspace/mlx_graphs_venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:339: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_r

Training mlx-graphs biased (p=1, q=0.5)  seed 2


/Users/vinay.pandya/mlx_graphs_workspace/mlx_graphs_venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:206: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/vinay.pandya/mlx_graphs_workspace/mlx_graphs_venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:206: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/vinay.pandya/mlx_graphs_workspace/mlx_graphs_venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:206: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/vinay.pandya/mlx_graphs_workspace/mlx_graphs_venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:339: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_r

In [9]:
def epoch_time(runs):
    """Median seconds per epoch over all seeds, skipping each run's first epoch (compilation, caches)."""
    steady = [t for times, _, _ in runs for t in (times[1:] if len(times) > 1 else times)]
    return statistics.median(steady)

def mean_std(values, fmt):
    if len(values) == 1:
        return format(values[0], fmt)
    return f"{format(statistics.mean(values), fmt)} ± {format(statistics.stdev(values), fmt)}"

def final_losses(runs):
    return [losses[-1] for _, losses, _ in runs]

def accuracies(runs):
    return [acc for _, _, acc in runs]

NOT_SUPPORTED = "not supported"
TRAINING_HEADER = ["Walks", "PyG s/epoch", "mlx-graphs s/epoch", "mlx-graphs vs PyG", "PyG final loss", "mlx-graphs final loss"]

training_rows = []
for label, _, _ in WALK_SETTINGS:
    mxg, pyg = results[(label, "mlx-graphs")], results.get((label, "PyG"))
    t_mxg = epoch_time(mxg)
    t_pyg = epoch_time(pyg) if pyg else None
    training_rows.append([
        label,
        f"{t_pyg:.2f}" if pyg else NOT_SUPPORTED,
        f"{t_mxg:.2f}",
        f"{t_pyg / t_mxg:.2f}x" if pyg else "n/a",
        mean_std(final_losses(pyg), ".4f") if pyg else NOT_SUPPORTED,
        mean_std(final_losses(mxg), ".4f"),
    ])

print(f"{len(SEEDS)} seeds, {NUM_EPOCHS} epochs each (losses: mean ± std over seeds)")
print_table(TRAINING_HEADER, training_rows)

3 seeds, 10 epochs each (losses: mean ± std over seeds)
| Walks               | PyG s/epoch   | mlx-graphs s/epoch | mlx-graphs vs PyG | PyG final loss  | mlx-graphs final loss |
|---------------------|---------------|--------------------|-------------------|-----------------|-----------------------|
| uniform (p=1, q=1)  | 0.33          | 0.15               | 2.14x             | 0.8286 ± 0.0008 | 0.8287 ± 0.0007       |
| biased (p=1, q=0.5) | not supported | 0.18               | n/a               | not supported   | 0.8498 ± 0.0004       |


## 3. Embedding quality

A logistic-regression classifier is trained on each run's embeddings using Cora's standard train split and scored on the test split (1,000 nodes, so a single score varies by roughly ±1.5 points from the test set alone). Accuracy is reported as mean ± std over `SEEDS`. If the two libraries' ranges overlap, they learn embeddings of similar quality and the speed comparison is fair.

In [10]:
QUALITY_HEADER = ["Walks", "PyG test accuracy", "mlx-graphs test accuracy"]

quality_rows = []
for label, _, _ in WALK_SETTINGS:
    pyg = results.get((label, "PyG"))
    quality_rows.append([
        label,
        mean_std(accuracies(pyg), ".3f") if pyg else NOT_SUPPORTED,
        mean_std(accuracies(results[(label, "mlx-graphs")]), ".3f"),
    ])

print(f"Mean ± std over {len(SEEDS)} seeds")
print_table(QUALITY_HEADER, quality_rows)

Mean ± std over 3 seeds
| Walks               | PyG test accuracy | mlx-graphs test accuracy |
|---------------------|-------------------|--------------------------|
| uniform (p=1, q=1)  | 0.719 ± 0.010     | 0.706 ± 0.007            |
| biased (p=1, q=0.5) | not supported     | 0.722 ± 0.010            |


## Summary

In [11]:
print(f"Machine: {platform.machine()}, mlx {pkg_version('mlx')}, mlx_cluster {pkg_version('mlx_cluster')}, "
      f"torch {pkg_version('torch')}, torch_geometric {pkg_version('torch_geometric')}, pyg-lib {pkg_version('pyg-lib')}\n")
print("Walk sampling (median ms per call):")
print_table(["Walks", "PyG CPU (ms)", "MLX CPU (ms)", "MLX GPU (ms)", "MLX GPU vs PyG"], sampling_rows)
print(f"\nSettings: {len(SEEDS)} seeds, {NUM_EPOCHS} epochs, MATCH_INIT={MATCH_INIT}, MATCH_ADAM={MATCH_ADAM}, "
      f"PyG device {TORCH_DEVICE}, mlx-graphs biased walks on GPU: {MXG_BIASED_WALKS_ON_GPU}")
print("\nTraining (median s per epoch over seeds, first epoch excluded; loss mean ± std):")
print_table(TRAINING_HEADER, training_rows)
print("\nEmbedding quality (logistic regression, Cora test split, mean ± std over seeds):")
print_table(QUALITY_HEADER, quality_rows)

Machine: arm64, mlx 0.32.4, mlx_cluster 0.1.2, torch 2.12.1, torch_geometric 2.8.1, pyg-lib 0.9.0+pt212

Walk sampling (median ms per call):
| Walks               | PyG CPU (ms)  | MLX CPU (ms) | MLX GPU (ms) | MLX GPU vs PyG |
|---------------------|---------------|--------------|--------------|----------------|
| uniform (p=1, q=1)  | 18.0          | 15.8         | 2.4          | 7.4x           |
| biased (p=1, q=0.5) | not supported | 52.4         | 6.1          | n/a            |

Settings: 3 seeds, 10 epochs, MATCH_INIT=True, MATCH_ADAM=True, PyG device mps, mlx-graphs biased walks on GPU: True

Training (median s per epoch over seeds, first epoch excluded; loss mean ± std):
| Walks               | PyG s/epoch   | mlx-graphs s/epoch | mlx-graphs vs PyG | PyG final loss  | mlx-graphs final loss |
|---------------------|---------------|--------------------|-------------------|-----------------|-----------------------|
| uniform (p=1, q=1)  | 0.33          | 0.15               | 2.14

### Reading the results

- **Walk sampling** isolates the kernels: this is where the Metal walk kernels in `mlx_cluster` are compared directly with pyg-lib's CPU sampler.
- **Training** includes everything else as well (embedding lookups, loss, optimizer), so the gap is usually smaller than for sampling alone.
- **Biased walks** have no PyG comparison: pyg-lib's `random_walk` only supports `p = q = 1`, so PyG 2.8's `Node2Vec` can't train biased node2vec. Those rows show mlx-graphs alone.
- **Accuracy** is only meaningfully different if the mean ± std ranges don't overlap; with a few seeds, differences of 1–2 points are usually noise.
- If `MXG_BIASED_WALKS_ON_GPU` is `False`, the biased-walk training row for mlx-graphs samples on the CPU, which is how upstream mlx-graphs currently behaves.
- With `MATCH_INIT = False`, PyG starts from much larger embeddings (std 1.0 vs 0.088), so its loss starts far higher and it needs more epochs to reach the same accuracy. That's a difference in defaults, not in the algorithm.
- The two libraries use different random number generators, so the same seed doesn't give the same walks across libraries; compare the mean ± std ranges, not single runs.